# 🧪 实验五：模型评估与推理

**MobileNetV3 图像分类实战教程**

---

## 实验目标

1. 加载训练好的模型 checkpoint，评估其在验证集上的性能
2. 对单张图像进行推理，理解前向传播和 Softmax 输出
3. 可视化预测结果，理解模型「学到了什么」
4. 对比 MobileNetV3-Large 和 Small 的精度与效率

---


In [ ]:
# ====== 1. 导入所需库 ======
# 先设置环境变量抑制警告，子进程（spawn）也会继承
import os
os.environ['PYTHONWARNINGS'] = 'ignore'

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# 导入 torch_npu（Ascend NPU 支持）
import torch_npu

# ====== 自动检测设备：NPU > CPU ======
def get_device():
    if torch.npu.is_available():
        return torch.device('npu:0')
    else:
        return torch.device('cpu')

device = get_device()
print(f"PyTorch 版本: {torch.__version__}")
print(f"使用设备: {device}")

import warnings
warnings.filterwarnings("ignore", message=".*owner does not match.*")
warnings.filterwarnings("ignore", message=".*TASK_QUEUE_ENABLE.*")
warnings.filterwarnings("ignore", message=".*Permission mismatch.*")

In [ ]:
# ====== MobileNetV3 完整模型定义（自包含，不依赖外部文件）======

def get_model_parameters(model):
    total_parameters = 0
    for layer in list(model.parameters()):
        layer_parameter = 1
        for l in list(layer.size()):
            layer_parameter *= l
        total_parameters += layer_parameter
    return total_parameters

def _make_divisible(v, divisor=8, min_value=None):
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v

def _weights_init(m):
    if isinstance(m, nn.Conv2d):
        torch.nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        m.weight.data.fill_(1)
        m.bias.data.zero_()
    elif isinstance(m, nn.Linear):
        n = m.weight.size(1)
        m.weight.data.normal_(0, 0.01)
        m.bias.data.zero_()

class h_sigmoid(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.inplace = inplace
    def forward(self, x):
        return F.relu6(x + 3., inplace=self.inplace) / 6.

class h_swish(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.inplace = inplace
    def forward(self, x):
        out = F.relu6(x + 3., self.inplace) / 6.
        return out * x

class SqueezeBlock(nn.Module):
    def __init__(self, exp_size, divide=4):
        super().__init__()
        self.dense = nn.Sequential(
            nn.Linear(exp_size, exp_size // divide),
            nn.ReLU(inplace=True),
            nn.Linear(exp_size // divide, exp_size),
            h_sigmoid()
        )
    def forward(self, x):
        batch, channels, height, width = x.size()
        out = F.avg_pool2d(x, kernel_size=[height, width]).view(batch, -1)
        out = self.dense(out).view(batch, channels, 1, 1)
        return out * x

class MobileBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernal_size, stride, nonLinear, SE, exp_size):
        super().__init__()
        padding = (kernal_size - 1) // 2
        self.use_connect = stride == 1 and in_channels == out_channels
        self.SE = SE
        activation = nn.ReLU if nonLinear == "RE" else h_swish
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, exp_size, 1, 1, 0, bias=False),
            nn.BatchNorm2d(exp_size), activation(inplace=True))
        self.depth_conv = nn.Sequential(
            nn.Conv2d(exp_size, exp_size, kernal_size, stride, padding, groups=exp_size),
            nn.BatchNorm2d(exp_size))
        if self.SE:
            self.squeeze_block = SqueezeBlock(exp_size)
        self.point_conv = nn.Sequential(
            nn.Conv2d(exp_size, out_channels, 1, 1, 0),
            nn.BatchNorm2d(out_channels), activation(inplace=True))
    def forward(self, x):
        out = self.depth_conv(self.conv(x))
        if self.SE:
            out = self.squeeze_block(out)
        out = self.point_conv(out)
        return x + out if self.use_connect else out

class MobileNetV3(nn.Module):
    def __init__(self, model_mode="LARGE", num_classes=1000, multiplier=1.0, dropout_rate=0.0):
        super().__init__()
        self.num_classes = num_classes
        if model_mode == "LARGE":
            layers = [
                [16, 16, 3, 1, "RE", False, 16], [16, 24, 3, 2, "RE", False, 64],
                [24, 24, 3, 1, "RE", False, 72], [24, 40, 5, 2, "RE", True, 72],
                [40, 40, 5, 1, "RE", True, 120], [40, 40, 5, 1, "RE", True, 120],
                [40, 80, 3, 2, "HS", False, 240], [80, 80, 3, 1, "HS", False, 200],
                [80, 80, 3, 1, "HS", False, 184], [80, 80, 3, 1, "HS", False, 184],
                [80, 112, 3, 1, "HS", True, 480], [112, 112, 3, 1, "HS", True, 672],
                [112, 160, 5, 1, "HS", True, 672], [160, 160, 5, 2, "HS", True, 672],
                [160, 160, 5, 1, "HS", True, 960],
            ]
            init_conv_out = _make_divisible(16 * multiplier)
            self.init_conv = nn.Sequential(
                nn.Conv2d(3, init_conv_out, 3, 2, 1), nn.BatchNorm2d(init_conv_out), h_swish())
            self.block = nn.Sequential(*[
                MobileBlock(_make_divisible(ic * multiplier), _make_divisible(oc * multiplier),
                           k, s, nl, se, _make_divisible(exp * multiplier))
                for ic, oc, k, s, nl, se, exp in layers])
            self.out_conv1 = nn.Sequential(
                nn.Conv2d(_make_divisible(160 * multiplier), _make_divisible(960 * multiplier), 1, 1),
                nn.BatchNorm2d(_make_divisible(960 * multiplier)), h_swish())
            self.out_conv2 = nn.Sequential(
                nn.Conv2d(_make_divisible(960 * multiplier), _make_divisible(1280 * multiplier), 1, 1),
                h_swish(), nn.Dropout(dropout_rate),
                nn.Conv2d(_make_divisible(1280 * multiplier), num_classes, 1, 1))
        elif model_mode == "SMALL":
            layers = [
                [16, 16, 3, 2, "RE", True, 16], [16, 24, 3, 2, "RE", False, 72],
                [24, 24, 3, 1, "RE", False, 88], [24, 40, 5, 2, "RE", True, 96],
                [40, 40, 5, 1, "RE", True, 240], [40, 40, 5, 1, "RE", True, 240],
                [40, 48, 5, 1, "HS", True, 120], [48, 48, 5, 1, "HS", True, 144],
                [48, 96, 5, 2, "HS", True, 288], [96, 96, 5, 1, "HS", True, 576],
                [96, 96, 5, 1, "HS", True, 576],
            ]
            init_conv_out = _make_divisible(16 * multiplier)
            self.init_conv = nn.Sequential(
                nn.Conv2d(3, init_conv_out, 3, 2, 1), nn.BatchNorm2d(init_conv_out), h_swish())
            self.block = nn.Sequential(*[
                MobileBlock(_make_divisible(ic * multiplier), _make_divisible(oc * multiplier),
                           k, s, nl, se, _make_divisible(exp * multiplier))
                for ic, oc, k, s, nl, se, exp in layers])
            self.out_conv1 = nn.Sequential(
                nn.Conv2d(_make_divisible(96 * multiplier), _make_divisible(576 * multiplier), 1, 1),
                SqueezeBlock(_make_divisible(576 * multiplier)),
                nn.BatchNorm2d(_make_divisible(576 * multiplier)), h_swish())
            self.out_conv2 = nn.Sequential(
                nn.Conv2d(_make_divisible(576 * multiplier), _make_divisible(1280 * multiplier), 1, 1),
                h_swish(), nn.Dropout(dropout_rate),
                nn.Conv2d(_make_divisible(1280 * multiplier), num_classes, 1, 1))
        self.apply(_weights_init)
    def forward(self, x):
        out = self.block(self.init_conv(x))
        out = self.out_conv1(out)
        b, c, h, w = out.size()
        return self.out_conv2(F.avg_pool2d(out, [h, w])).view(b, -1)

print("✅ MobileNetV3 模型定义加载完成")

---
## 2. 加载预训练模型

使用已经在 Tiny ImageNet 上训练了 300 个 epoch 的最佳模型。

In [ ]:
# ====== 2. 加载预训练模型 ======
NUM_CLASSES = 200
MODEL_MODE = "LARGE"  # 或 "SMALL"

# 创建模型结构
model = MobileNetV3(model_mode=MODEL_MODE, num_classes=NUM_CLASSES, multiplier=1.0, dropout_rate=0.0)
model = nn.DataParallel(model).to(device)

# 加载 checkpoint
ckpt_path = os.path.abspath(f'../checkpoint/best_model_{MODEL_MODE}_TINY_IMAGENET_ckpt.t7')

if os.path.exists(ckpt_path):
    checkpoint = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(checkpoint['model'])
    
    best_acc1 = checkpoint.get('best_acc1', 'N/A')
    best_acc5 = checkpoint.get('best_acc5', 'N/A')
    epoch = checkpoint.get('epoch', 'N/A')
    
    print(f"✅ 模型加载成功!")
    print(f"   来源: {ckpt_path}")
    print(f"   训练 epoch: {epoch}")
    print(f"   最佳 Acc@1: {best_acc1}")
    print(f"   最佳 Acc@5: {best_acc5}")
else:
    print(f"❌ 找不到 checkpoint 文件: {ckpt_path}")
    print("   请先运行完整训练，或使用已提供的 checkpoint")

---
## 3. 在验证集上评估模型

评估模型在整个验证集上的 Top-1 和 Top-5 准确率。

In [ ]:
# ====== 3. 准备验证集 DataLoader ======
DATA_DIR = os.path.abspath('tiny-imagenet-200')
BATCH_SIZE = 32
NUM_WORKERS = 1

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_dataset = ImageFolder(
    os.path.join(DATA_DIR, 'val'),
    transform=val_transform
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, multiprocessing_context='spawn', persistent_workers=True
)

print(f"验证集大小: {len(val_dataset):,} 张图像")
print(f"类别数: {len(val_dataset.classes)}")
print(f"Batch 数: {len(val_loader)}")

# 加载类别名称
wnids_path = os.path.join(DATA_DIR, 'wnids.txt')
words_path = os.path.join(DATA_DIR, 'words.txt')
with open(wnids_path, 'r') as f:
    class_ids = sorted([line.strip() for line in f.readlines()])
word_map = {}
with open(words_path, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 2:
            word_map[parts[0]] = parts[1]

class_names = [word_map.get(cid, cid) for cid in class_ids]

In [ ]:
# ====== 4. 评估模型 ======
def evaluate(model, loader, device):
    """评估模型准确率"""
    model.eval()
    correct1 = 0
    correct5 = 0
    total = 0
    
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            
            # Top-1 准确率
            _, pred = output.topk(1, 1, True, True)
            correct1 += pred.eq(target.view(-1, 1)).sum().item()
            
            # Top-5 准确率
            _, pred5 = output.topk(5, 1, True, True)
            correct5 += pred5.eq(target.view(-1, 1).expand_as(pred5)).sum().item()
            
            total += target.size(0)
    
    acc1 = 100.0 * correct1 / total
    acc5 = 100.0 * correct5 / total
    return acc1, acc5


print("正在评估模型，请稍候...")
acc1, acc5 = evaluate(model, val_loader, device)
print(f"\n{'='*50}")
print(f"MobileNetV3-{MODEL_MODE} on Tiny ImageNet")
print(f"{'='*50}")
print(f"  Top-1 准确率: {acc1:.2f}%")
print(f"  Top-5 准确率: {acc5:.2f}%")
print(f"  参数量: {get_model_parameters(model):,}")
print(f"{'='*50}")
print("💡 Top-1: 模型预测的第一名是否正确")
print("💡 Top-5: 正确类别是否在模型预测的前五名中")

---
## 4. 单张图像推理

使用训练好的模型对单张图像进行分类预测，并可视化预测结果。

In [ ]:
# ====== 5. 单张图像推理 ======
def predict_image(model, image_path, class_names, device, topk=5):
    """对单张图像进行预测并可视化"""
    # 图像预处理
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # 加载图像
    image = Image.open(image_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)
    
    # 预测
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        probabilities = torch.nn.functional.softmax(output, dim=1)
        topk_probs, topk_indices = torch.topk(probabilities, topk, dim=1)
    
    return image, topk_probs, topk_indices


def display_prediction(image, topk_probs, topk_indices, class_names):
    """显示预测结果"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # 显示图像
    ax1.imshow(image)
    ax1.axis('off')
    ax1.set_title('Input Image', fontsize=14)
    
    # 显示预测结果
    probs = topk_probs[0].cpu().numpy() * 100
    indices = topk_indices[0].cpu().numpy()
    labels = [class_names[idx] if idx < len(class_names) else f'Class {idx}' for idx in indices]
    
    colors = ['#2ecc71' if i == 0 else '#95a5a6' for i in range(len(probs))]
    bars = ax2.barh(range(len(probs)), probs[::-1], color=colors[::-1])
    ax2.set_yticks(range(len(probs)))
    ax2.set_yticklabels(labels[::-1])
    ax2.set_xlabel('Confidence (%)')
    ax2.set_title('Top-5 Predictions', fontsize=14)
    
    for bar, prob in zip(bars, probs[::-1]):
        ax2.text(prob + 0.5, bar.get_y() + bar.get_height()/2,
                f'{prob:.1f}%', va='center')
    
    plt.tight_layout()
    plt.show()
    
    print(f"🎯 预测结果: {labels[0]}")
    print(f"   置信度: {probs[0]:.2f}%")


# 从验证集中随机选取一张图像进行推理
import random

# 随机选一个类别
sample_class_idx = random.randint(0, len(class_ids) - 1)
sample_class_dir = os.path.join(DATA_DIR, 'val', class_ids[sample_class_idx])
sample_images = [f for f in os.listdir(sample_class_dir) if f.endswith('.JPEG')]
sample_image_path = os.path.join(sample_class_dir, random.choice(sample_images))

print(f"随机选取: {class_ids[sample_class_idx]} ({class_names[sample_class_idx]})")
print(f"图像路径: {sample_image_path}\n")

image, probs, indices = predict_image(model, sample_image_path, class_names, device)
display_prediction(image, probs, indices, class_names)

true_label = class_names[sample_class_idx]
pred_label = class_names[indices[0][0].item()]
print(f"\n{'✅ 预测正确!' if true_label == pred_label else '❌ 预测错误'}")
print(f"  真实类别: {true_label}")
print(f"  预测类别: {pred_label}")

In [ ]:
# ====== 6. 多张图像推理展示 ======
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
fig.suptitle('Multiple Predictions', fontsize=16)

for idx, ax in enumerate(axes.flat):
    # 随机选图像
    cid = random.choice(class_ids)
    img_dir = os.path.join(DATA_DIR, 'val', cid)
    img_files = [f for f in os.listdir(img_dir) if f.endswith('.JPEG')]
    if not img_files:
        continue
    img_path = os.path.join(img_dir, random.choice(img_files))
    
    # 预测
    image, probs, indices = predict_image(model, img_path, class_names, device, topk=1)
    pred_name = class_names[indices[0][0].item()]
    true_name = word_map.get(cid, cid)
    confidence = probs[0][0].item() * 100
    
    # 显示
    ax.imshow(image)
    color = 'green' if pred_name == true_name else 'red'
    ax.set_title(f'True:: {true_name}\nPred:: {pred_name} ({confidence:.0f}%)',
                fontsize=10, color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 5. 模型混淆分析（进阶）

分析模型在哪些类别上表现最好，哪些类别容易混淆。

In [ ]:
# ====== 7. 按类别统计准确率 ======
print("正在按类别统计准确率...")

class_correct = {i: 0 for i in range(NUM_CLASSES)}
class_total = {i: 0 for i in range(NUM_CLASSES)}

model.eval()
with torch.no_grad():
    for data, target in val_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        _, predicted = output.max(1)
        
        for i in range(target.size(0)):
            label = target[i].item()
            class_total[label] += 1
            if predicted[i].item() == label:
                class_correct[label] += 1

# 计算每类准确率
class_acc = {}
for i in range(NUM_CLASSES):
    if class_total[i] > 0:
        class_acc[i] = 100.0 * class_correct[i] / class_total[i]
    else:
        class_acc[i] = 0

# 显示 Top-5 最容易和最难的类别
sorted_classes = sorted(class_acc.items(), key=lambda x: x[1], reverse=True)

print(f"\n{'='*60}")
print(f"Top-5 最容易识别的类别:")
print(f"{'='*60}")
for rank, (cls, acc) in enumerate(sorted_classes[:5]):
    print(f"  {rank+1}. {class_names[cls]:30s} Acc@1: {acc:.1f}%")

print(f"\n{'='*60}")
print(f"Top-5 最难识别的类别:")
print(f"{'='*60}")
for rank, (cls, acc) in enumerate(sorted_classes[-5:]):
    print(f"  {rank+1}. {class_names[cls]:30s} Acc@1: {acc:.1f}%")

---
## 6. 模型对比：Large vs Small

对比 MobileNetV3-Large 和 Small 在参数量和准确率上的权衡。

In [ ]:
# ====== 8. Large vs Small 对比 ======
print(f"{'='*60}")
print(f"MobileNetV3 模型对比")
print(f"{'='*60}")
print(f"{'模型':<20} {'参数量':<12} {'Blocks':<10} {'Best Acc@1':<12}")
print(f"{'-'*60}")

for mode in ["LARGE", "SMALL"]:
    # 创建模型
    m = MobileNetV3(model_mode=mode, num_classes=NUM_CLASSES, multiplier=1.0, dropout_rate=0.0)
    params = get_model_parameters(m)
    
    # 计算 blocks
    if mode == "LARGE":
        n_blocks = 15
    else:
        n_blocks = 11
    
    # 尝试加载 checkpoint
    ckpt = os.path.abspath(f'../checkpoint/best_model_{mode}_TINY_IMAGENET_ckpt.t7')
    if os.path.exists(ckpt):
        ckpt_data = torch.load(ckpt, map_location='cpu')
        acc1 = ckpt_data.get('best_acc1', 'N/A')
    else:
        acc1 = 'N/A'
    
    print(f"{f'MobileNetV3-{mode}':<20} {params:<12,} {n_blocks:<10} {str(acc1):<12}")

print(f"{'='*60}")
print("\n💡 选择建议:")
print("  - 精度优先 → MobileNetV3-Large")
print("  - 速度/内存优先 → MobileNetV3-Small")
print("  - 通过 --multiplier 参数可以进一步调整模型宽度")

---
## 7. 命令行推理

本项目也提供了命令行推理工具 `inference.py`，可以直接在终端使用：

```bash
# 使用 Tiny ImageNet 模型对单张图像推理
python inference.py --image test.JPEG --model-mode LARGE --dataset-mode TINY_IMAGENET

# 指定 checkpoint 路径
python inference.py --image test.JPEG --model-mode LARGE --dataset-mode TINY_IMAGENET --checkpoint ./checkpoint/best_model_LARGE_TINY_IMAGENET_ckpt.t7

# 显示 Top-10 结果
python inference.py --image test.JPEG --model-mode LARGE --dataset-mode TINY_IMAGENET --topk 10
```

示例输出：
```
使用设备: npu:0
图像尺寸: (64, 64)
加载模型: ./checkpoint/best_model_LARGE_TINY_IMAGENET_ckpt.t7
模型训练精度: Acc@1=61.47, Acc@5=81.52

预测结果 (Top-5):
==================================================
  1. n02124075  ( 32)   置信度: 85.32%
  2. n04507155  ( 18)   置信度: 6.15%
  3. n02279972  ( 13)   置信度: 3.21%
  4. n07875152  ( 23)   置信度: 2.10%
  5. n07753592  ( 46)   置信度: 1.05%
==================================================
预测类别: n02124075
```

> 类别 ID 可在 `data/tiny-imagenet-200/words.txt` 中查找对应名称。
>
> **注意**：在 NPU 上运行 inference.py 时，需要先 `import torch_npu`，
> 或修改 inference.py 中的设备检测逻辑。

---
## 📝 实验小结

在本实验中，我们完成了：

1. ✅ 加载了预训练的 MobileNetV3 checkpoint
2. ✅ 在完整验证集上评估了 Top-1 和 Top-5 准确率
3. ✅ 对单张图像进行推理，可视化预测结果和置信度
4. ✅ 分析了每个类别的识别准确率
5. ✅ 对比了 Large 和 Small 模型的性能差异
6. ✅ 了解了命令行推理工具的使用方法

**课程总结：**

通过这 5 个实验，我们完整走过了 MobileNetV3 图像分类的整个流程：

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">实验</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">内容</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">核心技能</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">1️⃣ 环境准备与数据探索</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">数据集分析、数据增强</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">PyTorch DataLoader</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">2️⃣ 网络结构解析</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">MobileBlock、SE、h-swish</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">模型定义与参数量计算</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">3️⃣ 训练流程解读</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">CosineLR、EMA、标签平滑</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">训练技巧与日志分析</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">4️⃣ 模型训练实战</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">完整训练流程</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">训练、保存、恢复</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">5️⃣ 模型评估与推理</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">评估、推理、可视化</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">模型部署与应用</td>
    </tr>
  </tbody>
</table>

---
**🎉 恭喜你完成了所有实验！**

## 课后练习

1. (单选题) logits 形状为 (256, 200)，torch.topk(logits, 5, dim=1) 返回的两个张量形状是？
   - A. values(256,5) 与 indices(256,5)
   - B. values(5,256) 与 indices(5,256)
   - C. 只有一个张量 (256,5)
   - D. (256,200) 与 (256,200)

2. (单选题) Top-5 准确率的计算方式是？
   - A. 真实类别是否出现在模型概率最高的前 5 个类别中
   - B. 前 5 个样本预测正确
   - C. 5 次推理取平均
   - D. 类别数不超过 5

3. (多选题) 为保证评估结果可复现，需要固定？
   - A. 与训练一致的 Normalize/尺寸
   - B. 同一份 checkpoint 与 dtype
   - C. 固定随机种子（若推理含随机操作）
   - D. 训练集增强

4. (多选题) 混淆矩阵可以用于？
   - A. 发现高频混淆类别对
   - B. 计算每类召回率
   - C. 定位具体错误样本
   - D. 替代 Top-1 指标

5. (判断题) 同一个模型在相同数据上，Top-5 准确率一定不低于 Top-1。

6. (判断题) 推理时应沿用训练时的 RandAugment 以保证数据分布一致。

7. (填空题) torch.max(logits, dim=1) 返回两个张量：最大概率值与对应的 ____。

8. (填空题) 从 checkpoint 恢复推理前，必须先按相同结构创建模型，再调用 ____ 加载权重。

9. (简答题) 为什么批量评估通常比逐张图片评估更高效？请结合算子调度与访存说明。

10. (简答题) 如何使用混淆矩阵分析某一类别被系统性地误判为另一类别的问题？

11. (代码设计题) 编写 evaluate(model, loader, device)，返回 acc1、acc5 与宏平均召回率，要求 eval/no_grad。

12. (单选题) 评估时 logits 中出现 NaN，最合理的处理是？
   - A. 先检查 loss/预处理/数值稳定性，修复后再评估
   - B. 直接取 NaN 对应索引
   - C. 忽略 NaN
   - D. 将 NaN 替换为 0

13. (多选题) 类别不均衡时，更适合使用的指标包括？
   - A. 宏平均召回率
   - B. 各类 F1
   - C. 只看 Top-1
   - D. 混淆矩阵

14. (判断题) 计算验证准确率时应将模型保持为 train 模式，避免 BN 统计量偏差。

15. (简答题) 在 Tiny ImageNet 中，为什么 Top-5 对相似类别任务更友好？请结合类别粒度说明。

> 参考答案见 answer/02.07_evaluation_and_inference_answer.ipynb。